# 🍜 SF Restaurant Health Inspections - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests **San Francisco Department of Public Health inspections** of restaurants and other food businesses (2024-present) into Elasticsearch: every inspection, its result (**Pass**, **Conditional Pass**, or **Closure**), and the violations the inspector wrote up.

**What this notebook does:**
1. Fetches inspection records from [DataSF](https://data.sf.gov/d/tvy3-wexg) - no API key required
2. Normalises each record (dates, counts, neighborhood, `geo_point`)
3. Bulk-indexes everything into `sf_health_inspections`

---
> **Data source:** Health Inspection Scores (2024-Present), dataset `tvy3-wexg`. One row per inspection. The `violations` text lists the code sections cited, followed by the inspector-facing explanation - rich material for semantic search.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from datetime import date
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the inspections
The whole dataset is ~23k rows, so we take all of it. (A few rows carry typo'd future dates, so we cap at today.)

In [ ]:
SAMPLE_SIZE = 50000

def fetch_socrata(dataset_id, limit, page_size=10000, **params):
    """Page through a DataSF dataset with $limit/$offset until `limit` rows or the end."""
    url = f'https://data.sf.gov/resource/{dataset_id}.json'
    rows = []
    while len(rows) < limit:
        page = {**params, '$limit': min(page_size, limit - len(rows)), '$offset': len(rows)}
        r = requests.get(url, params=page, timeout=120)
        r.raise_for_status()
        batch = r.json()
        rows.extend(batch)
        print(f'  fetched {len(rows)} rows...', end='\r', flush=True)
        if len(batch) < page['$limit']:
            break
    return rows

rows = fetch_socrata('tvy3-wexg', SAMPLE_SIZE, **{
    '$where': f"inspection_date <= '{date.today().isoformat()}'",
    '$order': 'inspection_date DESC',
})
print(f'\n✅ Fetched {len(rows)} inspections')
print()
print('Sample row fields:')
for k, v in list(rows[0].items())[:14]:
    print(f'  {k:28s} {str(v)[:60]}')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_health_inspections'

mapping = {'mappings': {'properties': {
    'permit_number':      {'type': 'keyword'},   # stable id for the business
    'name':               {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'permit_type':        {'type': 'keyword'},   # e.g. H26R - RESTAURANT OVER 2,000 SQFT
    'address':            {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'inspection_date':    {'type': 'date'},
    'inspection_type':    {'type': 'keyword'},
    'result':             {'type': 'keyword'},   # Pass / Conditional Pass / Closure
    'violation_count':    {'type': 'integer'},
    'violations':         {'type': 'text'},      # code sections + explanations
    'suspension_notes':   {'type': 'text'},
    'neighborhood':       {'type': 'keyword'},
    'supervisor_district':{'type': 'keyword'},
    'location':           {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
The enriched index can answer questions like:
- *"When was NOVELA last inspected, and what did they get written up for?"*
- *"Which neighborhoods had the most closures this year?"*
- *"Find inspections that mention cooling food too slowly."*

In [ ]:
def geo_point(lat, lon):
    """Build a geo_point from lat/lon strings, skipping blanks and 0,0."""
    try:
        lat_f, lon_f = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if lat_f == 0.0 or lon_f == 0.0:
        return None
    return {'lat': lat_f, 'lon': lon_f}

def clean_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

def enrich(r):
    doc = {
        'permit_number':      r.get('permit_number'),
        'name':               (r.get('dba') or '').strip(),
        'permit_type':        r.get('permit_type'),
        'address':            ' '.join((r.get('street_address_clean') or r.get('street_address') or '').split()),
        'inspection_date':    r.get('inspection_date'),
        'inspection_type':    r.get('inspection_type'),
        'result':             r.get('facility_rating_status'),
        'violation_count':    clean_int(r.get('violation_count')),
        'violations':         r.get('violation_codes'),
        'suspension_notes':   r.get('suspension_notes'),
        'neighborhood':       r.get('analysis_neighborhood'),
        'supervisor_district':r.get('supervisor_district'),
        'location':           geo_point(r.get('latitude'), r.get('longitude')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify: quick sense-check queries

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'track_total_hits': True, 'aggs': {
    'by_result': {'terms': {'field': 'result', 'size': 5}},
    'closures_by_neighborhood': {
        'filter': {'term': {'result': 'Closure'}},
        'aggs': {'n': {'terms': {'field': 'neighborhood', 'size': 8}}},
    },
}})
print(f"Total inspections: {resp['hits']['total']['value']}\n")
print('By result:')
for b in resp['aggregations']['by_result']['buckets']:
    print(f"  {b['key']:18s} {b['doc_count']}")
print('\nClosures by neighborhood:')
for b in resp['aggregations']['closures_by_neighborhood']['n']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")

In [ ]:
# Full-text search over the violation text
resp = es.search(index=INDEX, body={'size': 5,
    'query': {'match': {'violations': 'rodent droppings'}},
    '_source': ['name', 'address', 'inspection_date', 'result'],
})
print(f"Inspections mentioning rodent droppings: {resp['hits']['total']['value']}")
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['inspection_date'][:10]}  {s['result']:16s} {s['name']} - {s.get('address', '')}")

## 6. Going further with Mistral

- **Semantic search:** back a `semantic_text` field on `violations` with a Mistral embedding endpoint and search by meaning ("pest problems", "food kept too warm"). See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **Plain-language verdicts:** pull a restaurant's last few inspections and ask a Mistral chat model to explain them in one friendly sentence, in the diner's language.
- **Owner helper:** for a business with a Conditional Pass, have Mistral turn the cited code sections into a checklist of fixes before re-inspection.
- **Combine:** join on name/address with [sf_business_locations](sf_business_locations.ipynb) to see how long a place has been open.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
Your `sf_health_inspections` index is ready and verified.

**To refresh the data**, re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.